# Advanced Time Series Visualisation

Standard line plots are the workhorse of time series analysis, but they only
tell part of the story. This notebook introduces advanced visualisation
techniques that reveal **relationships between series**, **seasonal structure**,
**distributional properties**, and **composition over time**.

We will cover:
1. Scatterplots between time series (with time as colour)
2. Seasonal heatmaps (month × year)
3. Box plots by season
4. Dual-axis plots (and their pitfalls)
5. Area charts and stacked plots
6. Publication-quality styling

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

%matplotlib inline

DATA_DIR = Path("../data")

print(f"pandas  {pd.__version__}")
print(f"seaborn {sns.__version__}")

### Load the datasets

In [ ]:
# Airline passengers (monthly, 1949-1960)
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    parse_dates=["Month"],
    index_col="Month",
)
airline.columns = ["Passengers"]
airline.index.freq = "MS"

# Energy production index (monthly, 1970-2018)
energy = pd.read_csv(
    DATA_DIR / "EnergyProduction.csv",
    parse_dates=["DATE"],
    index_col="DATE",
)
energy.index.freq = "MS"

# Alcohol sales (monthly, 1992-2019)
alcohol = pd.read_csv(
    DATA_DIR / "Alcohol_Sales.csv",
    parse_dates=["DATE"],
    index_col="DATE",
)
alcohol.columns = ["Sales"]
alcohol.index.freq = "MS"

# Pet-food price indices (monthly, 1985-2018)
cat_food = pd.read_csv(
    DATA_DIR / "PriceIndexCatFood.csv",
    parse_dates=["Date"],
    index_col="Date",
)
dog_food = pd.read_csv(
    DATA_DIR / "PriceIndexDogFood.csv",
    parse_dates=["Date"],
    index_col="Date",
)

print("All datasets loaded.")

In [ ]:
# Quick sanity check
for name, df in [("airline", airline), ("energy", energy),
                  ("alcohol", alcohol), ("cat_food", cat_food),
                  ("dog_food", dog_food)]:
    print(f"{name:>10}: {df.shape[0]:>4} rows, "
          f"{df.index.min().date()} to {df.index.max().date()}")

---
## 1. Scatterplots Between Time Series

When two series share the same time index we can plot them against each other.
If we colour the points by date we get a sense of how the relationship
evolves over time.

In [ ]:
# Merge cat-food and dog-food price indices on their shared dates
pet = cat_food.join(dog_food, how="inner")
pet.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

# Convert dates to numeric values for the colormap
time_numeric = mdates.date2num(pet.index.to_pydatetime())

scatter = ax.scatter(
    pet["CatFood"],
    pet["DogFood"],
    c=time_numeric,
    cmap="viridis",
    edgecolors="k",
    linewidths=0.3,
    alpha=0.8,
    s=30,
)

# Colour bar with readable date labels
cbar = fig.colorbar(scatter, ax=ax)
loc = mdates.AutoDateLocator()
cbar.ax.yaxis.set_major_locator(loc)
cbar.ax.yaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
cbar.set_label("Date")

ax.set_xlabel("Cat Food Price Index")
ax.set_ylabel("Dog Food Price Index")
ax.set_title("Cat-Food vs Dog-Food Price Index (coloured by time)")
plt.tight_layout()
plt.show()

### Correlation coefficient

In [ ]:
r = pet["CatFood"].corr(pet["DogFood"])
print(f"Pearson correlation: {r:.4f}")

> **Caution — correlation is not causation.**
> Both price indices are driven by common macroeconomic factors
> (inflation, commodity prices, supply-chain costs). A high correlation
> between two time series is often the result of a **shared trend** rather
> than a direct causal link. Always consider confounding variables and
> apply differencing or detrending before drawing conclusions.

---
## 2. Seasonal Heatmaps

A month-by-year heatmap displays both **trend** (left to right) and
**seasonality** (top to bottom) in a single image.

### 2a. Airline passengers

In [ ]:
# Build a month × year pivot table
airline_pivot = airline.copy()
airline_pivot["Year"] = airline_pivot.index.year
airline_pivot["Month"] = airline_pivot.index.month

pivot_air = airline_pivot.pivot_table(
    values="Passengers", index="Month", columns="Year"
)
pivot_air

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

sns.heatmap(
    pivot_air,
    cmap="YlOrRd",
    annot=True,
    fmt=".0f",
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Airline Passengers — Monthly Heatmap")
ax.set_ylabel("Month")
ax.set_xlabel("Year")
plt.tight_layout()
plt.show()

Reading the heatmap:
- **Left → right**: colours get warmer, confirming the upward trend.
- **Top → bottom**: months 6–9 (summer) are consistently the warmest band,
  showing the seasonal peak.

### 2b. Energy production

In [ ]:
energy_pivot = energy.copy()
energy_pivot["Year"] = energy_pivot.index.year
energy_pivot["Month"] = energy_pivot.index.month

pivot_energy = energy_pivot.pivot_table(
    values="EnergyIndex", index="Month", columns="Year"
)

fig, ax = plt.subplots(figsize=(20, 6))
sns.heatmap(
    pivot_energy,
    cmap="coolwarm",
    linewidths=0.3,
    ax=ax,
)
ax.set_title("Energy Production Index — Monthly Heatmap")
ax.set_ylabel("Month")
ax.set_xlabel("Year")
plt.tight_layout()
plt.show()

Energy production shows a **bimodal** seasonal pattern: peaks in both
winter (heating) and summer (cooling), with troughs in spring and autumn.

---
## 3. Box Plots by Season

Monthly box plots summarise the **distribution** of values for each calendar
month. They show the median, interquartile range, and outliers, making it
easy to spot which months have the highest values and the most variability.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.boxplot(
    x=airline.index.month,
    y=airline["Passengers"],
    palette="Set2",
    ax=ax,
)

ax.set_xlabel("Month")
ax.set_ylabel("Thousands of Passengers")
ax.set_title("Airline Passengers — Distribution by Month")
plt.tight_layout()
plt.show()

Observations:
- July and August have the highest medians — the summer travel peak.
- The boxes grow wider for later months because the upward trend inflates
  the range (values from later years are larger).
- November is a relative low point.

In [ ]:
# Same idea for alcohol sales
fig, ax = plt.subplots(figsize=(10, 5))

sns.boxplot(
    x=alcohol.index.month,
    y=alcohol["Sales"],
    palette="muted",
    ax=ax,
)

ax.set_xlabel("Month")
ax.set_ylabel("Sales (millions USD)")
ax.set_title("Alcohol Sales — Distribution by Month")
plt.tight_layout()
plt.show()

December stands out with by far the highest alcohol sales — the holiday
effect is unmistakable.

---
## 4. Dual-Axis Plots

Sometimes we want to overlay two series that have **different units or
scales** on the same time axis. Matplotlib's `twinx()` creates a second
y-axis on the right.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

colour_left = "tab:blue"
colour_right = "tab:red"

# Left axis — airline passengers
ax1.plot(airline.index, airline["Passengers"], color=colour_left, label="Airline Passengers")
ax1.set_xlabel("Date")
ax1.set_ylabel("Thousands of Passengers", color=colour_left)
ax1.tick_params(axis="y", labelcolor=colour_left)

# Right axis — we'll create a synthetic "fuel cost" series for illustration
# Here we use the energy production index clipped to the airline date range
# Since energy starts in 1970 and airline ends in 1960, let's use cat/dog food instead
ax2 = ax1.twinx()
ax2.plot(cat_food.index, cat_food["CatFood"], color=colour_right, alpha=0.8, label="Cat Food Price Index")
ax2.set_ylabel("Price Index", color=colour_right)
ax2.tick_params(axis="y", labelcolor=colour_right)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

ax1.set_title("Dual-Axis Plot — Two Unrelated Series on the Same Time Axis")
plt.tight_layout()
plt.show()

> **When are dual-axis plots appropriate?**
>
> Dual-axis charts can be **misleading** because:
> - The two y-axes can be independently rescaled, making any two series
>   appear to move together.
> - Readers may incorrectly assume a causal link.
>
> Use them only when:
> - The two series are **genuinely related** (e.g., temperature and energy demand).
> - You make the **axis colours** match the line colours.
> - You explain the relationship in the surrounding text.
>
> Often a better alternative is to **normalise** both series to a common
> scale (e.g., z-scores or percent change) and plot on a single axis.

In [ ]:
# Better alternative: normalise to z-scores and plot on a single axis
pet_z = pet.apply(lambda s: (s - s.mean()) / s.std())

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(pet_z.index, pet_z["CatFood"], label="Cat Food (z-score)")
ax.plot(pet_z.index, pet_z["DogFood"], label="Dog Food (z-score)")
ax.set_ylabel("Standardised Value")
ax.set_title("Normalised Comparison — Single Axis")
ax.legend()
ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
plt.tight_layout()
plt.show()

---
## 5. Area Charts

Area charts fill the space between the line and the x-axis, giving a
stronger visual impression of **magnitude**.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.fill_between(
    airline.index, airline["Passengers"],
    alpha=0.4, color="steelblue",
)
ax.plot(airline.index, airline["Passengers"], color="steelblue", linewidth=1)
ax.set_title("Airline Passengers — Area Chart")
ax.set_ylabel("Thousands of Passengers")
plt.tight_layout()
plt.show()

### Stacked area chart

When you have several components that sum to a total, a stacked area chart
shows composition over time.

In [ ]:
# Simulate quarterly components from the airline data for illustration
np.random.seed(42)
passengers = airline["Passengers"].values

# Split passengers into three fictional route groups
domestic_share = 0.45 + 0.05 * np.random.randn(len(passengers))
europe_share = 0.35 + 0.03 * np.random.randn(len(passengers))
other_share = 1.0 - domestic_share - europe_share

components = pd.DataFrame({
    "Domestic": passengers * domestic_share,
    "Europe": passengers * europe_share,
    "Other": passengers * other_share,
}, index=airline.index)

fig, ax = plt.subplots(figsize=(12, 5))
components.plot.area(ax=ax, alpha=0.7, cmap="Set2")
ax.set_title("Stacked Area Chart — Fictional Route Breakdown")
ax.set_ylabel("Thousands of Passengers")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

---
## 6. Publication-Quality Styling

Matplotlib ships with many built-in styles. Choosing the right one (and
tweaking a few `rcParams`) can turn a notebook chart into a
publication-ready figure.

### 6a. Built-in styles

In [ ]:
styles_to_show = ["seaborn-v0_8-whitegrid", "ggplot", "bmh"]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for ax, style_name in zip(axes, styles_to_show):
    with plt.style.context(style_name):
        ax.plot(airline.index, airline["Passengers"])
        ax.set_title(f"Style: {style_name}", fontsize=11)
        ax.set_ylabel("Passengers")

plt.suptitle("Comparing Matplotlib Styles", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 6b. Custom rcParams

In [ ]:
# Temporarily override rcParams for a polished look
custom_rc = {
    "figure.dpi": 120,
    "font.size": 11,
    "font.family": "sans-serif",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "lines.linewidth": 1.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
}

with plt.rc_context(custom_rc):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(airline.index, airline["Passengers"], color="#2c7bb6")
    ax.set_title("Airline Passengers — Publication Style")
    ax.set_ylabel("Thousands of Passengers")
    ax.set_xlabel("")
    plt.tight_layout()
    plt.show()

### 6c. Colour-blind-friendly palettes

In [ ]:
# The 'colorblind' palette from seaborn is specifically designed for
# accessibility. Alternatively, use the 'tab10' palette or choose
# colours from ColorBrewer.

cb_palette = sns.color_palette("colorblind", n_colors=5)
sns.palplot(cb_palette)
plt.title("Seaborn 'colorblind' palette")
plt.show()

In [ ]:
# Apply the colour-blind palette to a multi-line plot
with sns.color_palette("colorblind"):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(pet.index, pet["CatFood"], label="Cat Food")
    ax.plot(pet.index, pet["DogFood"], label="Dog Food")
    ax.set_title("Price Indices — Colour-Blind-Friendly Palette")
    ax.set_ylabel("Price Index")
    ax.legend()
    plt.tight_layout()
    plt.show()

### 6d. Saving figures

In [ ]:
# Demonstrate saving a figure (writes to a temporary location)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(airline.index, airline["Passengers"], color="#2c7bb6")
ax.set_title("Airline Passengers")
ax.set_ylabel("Thousands of Passengers")
plt.tight_layout()

# Save at 300 DPI with a tight bounding box
save_path = Path("/tmp/airline_passengers_pub.png")
fig.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Figure saved to {save_path}  ({save_path.stat().st_size / 1024:.1f} KB)")
plt.show()

---
## Summary

| Technique | Best For | Key Function |
|---|---|---|
| **Scatterplot (colour = time)** | Relationships between two series | `ax.scatter(..., c=time)` |
| **Seasonal heatmap** | Trend + seasonality at a glance | `sns.heatmap(pivot)` |
| **Box plot by month** | Distribution per season | `sns.boxplot(x=month, y=values)` |
| **Dual-axis plot** | Overlaying different scales | `ax.twinx()` |
| **Area / stacked area** | Magnitude and composition | `df.plot.area()` |
| **Style + rcParams** | Publication-ready output | `plt.style.use()`, `plt.rc_context()` |

**Key takeaways:**
- Always match the visualisation to the question you are asking.
- Dual-axis plots should be used sparingly and with clear labelling.
- Colour-blind-friendly palettes make your work accessible to a wider audience.
- `fig.savefig(dpi=300, bbox_inches='tight')` produces print-quality output.